# Basic SD 1.5 Inpainting Image/Mask Test

This notebook intentionally ignores the rest of the repo.

It tests only:

- one selected source image;
- one selected mask image;
- the base `stable-diffusion-v1-5/stable-diffusion-inpainting` model.

No LoRA. No backend. No frontend. No GeoJSON. No crop logic. No dataset script.

In [ ]:
from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import DPMSolverMultistepScheduler, StableDiffusionInpaintPipeline
from PIL import Image, ImageOps


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "backend").exists() and (candidate / "frontend").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not find repo root. Run this notebook from the repo or notebooks directory.")


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "outputs" / "notebook_baseline_inpaint"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d_%H%M%S")

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"
DATASET_NAME = "trees"
DATASET_DIR = REPO_ROOT / "datasets" / DATASET_NAME
LORA_DIR = REPO_ROOT / "outputs" / f"lora_{DATASET_NAME}"

# Leave these as None to use the first image/mask pair from DATASET_DIR.
IMAGE_PATH = None
MASK_PATH = None

# Stable Diffusion inpainting expects white pixels to be repainted and black pixels to be preserved.
INVERT_MASK = False

# Keep True if the model already exists in your Hugging Face cache. Set False only for first download.
LOCAL_FILES_ONLY = True

PROMPT = "satellite view of trees, canopy cover, urban vegetation"
NEGATIVE_PROMPT = "blurry, distorted, low quality, cartoon, warped perspective, repeated artifacts"

# Set to an integer for repeatable results. Set to None for a new result each run.
SEED = None
RUN_SEED = int(SEED) if SEED is not None else int(time.time()) % 2_147_483_647
print("repo root:", REPO_ROOT)
print("dataset:", DATASET_DIR)
print("output:", OUTPUT_DIR)
print("run id:", RUN_ID)
print("seed:", RUN_SEED)
RESOLUTION = 512
NUM_INFERENCE_STEPS = 40
GUIDANCE_SCALE = 6.5
STRENGTH = 1.0


## Load And Inspect Image/Mask

This cell checks the basics before any model is loaded. If the mask is wrong here, the model result will also be wrong.

Important: white mask pixels are the area Stable Diffusion will repaint.

In [ ]:
def first_dataset_pair(dataset_dir: Path) -> tuple[Path, Path]:
    metadata_path = dataset_dir / "metadata.jsonl"
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Missing dataset metadata: {metadata_path}\n"
            "Prepare a dataset first, for example:\n"
            "  cd notebooks\n"
            "  uv run python scripts/prepare_wroclaw_trees_lora_dataset.py --output-dir ../datasets/trees"
        )

    for line in metadata_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            row = json.loads(line)
            return dataset_dir / row["image"], dataset_dir / row["mask"]

    raise ValueError(f"No records found in {metadata_path}")


if IMAGE_PATH is None or MASK_PATH is None:
    image_path, mask_path = first_dataset_pair(DATASET_DIR)
else:
    image_path = Path(IMAGE_PATH)
    mask_path = Path(MASK_PATH)

if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path}")
if not mask_path.exists():
    raise FileNotFoundError(f"Mask not found: {mask_path}")

raw_image = Image.open(image_path)
raw_mask = Image.open(mask_path)

print("image path:", image_path)
print("mask path: ", mask_path)
print("image mode/size:", raw_image.mode, raw_image.size)
print("mask mode/size: ", raw_mask.mode, raw_mask.size)

if raw_image.size != raw_mask.size:
    print("WARNING: image and mask have different sizes. The mask will be resized to match the image before inference.")


In [ ]:
image = raw_image.convert("RGB")
mask = raw_mask.convert("L")

if mask.size != image.size:
    mask = mask.resize(image.size, Image.Resampling.NEAREST)

if INVERT_MASK:
    mask = ImageOps.invert(mask)

# Force a binary mask so semi-transparent or antialiased edges do not hide what is being tested.
mask = mask.point(lambda value: 255 if value > 127 else 0)

mask_array = np.asarray(mask)
coverage = float((mask_array > 0).mean())
unique_values = np.unique(mask_array)

print("binary mask values:", unique_values.tolist())
print(f"white/inpaint coverage: {coverage:.4%}")

if coverage == 0:
    raise ValueError("Mask has no white pixels. Nothing will be inpainted.")
if coverage == 1:
    raise ValueError("Mask is fully white. The entire image will be replaced.")

# Resize both together for a simple full-frame model test.
test_image = image.resize((RESOLUTION, RESOLUTION), Image.Resampling.BILINEAR)
test_mask = mask.resize((RESOLUTION, RESOLUTION), Image.Resampling.NEAREST)
test_mask = test_mask.point(lambda value: 255 if value > 127 else 0)

preserved_preview = Image.composite(test_image, Image.new("RGB", test_image.size, (0, 0, 0)), test_mask)
repaint_preview = Image.composite(Image.new("RGB", test_image.size, (255, 255, 255)), test_image, test_mask)

test_image.save(OUTPUT_DIR / "input_512.png")
test_mask.save(OUTPUT_DIR / "mask_512_white_repaints.png")
preserved_preview.save(OUTPUT_DIR / "preserved_area_preview.png")
repaint_preview.save(OUTPUT_DIR / "repaint_area_preview.png")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
items = [
    ("input", test_image, None),
    ("mask: white repaints", test_mask, "gray"),
    ("black = preserved", preserved_preview, None),
    ("white = repainted", repaint_preview, None),
]

for axis, (title, item, cmap) in zip(axes, items):
    axis.imshow(item, cmap=cmap)
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()

## Load Only The Base Inpainting Model

This must report `unet input channels: 9`. If it reports 4, the non-inpainting model was loaded.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,
    local_files_only=LOCAL_FILES_ONLY,
).to(device)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="dpmsolver++",
    use_karras_sigmas=True,
)

if device == "cuda":
    pipe.enable_attention_slicing()
    pipe.vae.enable_slicing()

pipe.set_progress_bar_config(disable=False)

print("device:", device)
print("pipeline:", pipe.__class__.__name__)
print("unet input channels:", pipe.unet.config.in_channels)

if int(pipe.unet.config.in_channels) != 9:
    raise RuntimeError("Wrong model loaded. Stable Diffusion inpainting must use a 9-channel UNet.")

## Run Inpainting

In [ ]:
generator = torch.Generator(device=device).manual_seed(RUN_SEED)

result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=test_image,
    mask_image=test_mask,
    height=RESOLUTION,
    width=RESOLUTION,
    strength=STRENGTH,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
).images[0]

result.save(OUTPUT_DIR / "latest_base_result.png")
result.save(OUTPUT_DIR / f"base_result_{RUN_ID}.png")
result

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
items = [
    ("input", test_image, None),
    ("mask", test_mask, "gray"),
    ("repaint area", repaint_preview, None),
    ("result", result, None),
]

for axis, (title, item, cmap) in zip(axes, items):
    axis.imshow(item, cmap=cmap)
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
print("saved outputs to:", OUTPUT_DIR)

## Optional LoRA Test

This section still ignores the repo scripts. It uses the same selected image, selected mask, prompt, seed, and base pipeline, then attaches one LoRA folder directly.

If the base result works but this result is poor, the problem is likely the LoRA weights, LoRA scale, training setup, or prompt fit.

In [ ]:
# Set this to the LoRA folder you want to test, or None to skip.
# The folder should contain pytorch_lora_weights.safetensors.
LORA_SCALE = 1.0

print("LoRA dir:", LORA_DIR)
if LORA_DIR is not None:
    print("LoRA weights exist:", (Path(LORA_DIR) / "pytorch_lora_weights.safetensors").exists())


In [ ]:
def load_base_pipe_with_optional_lora(lora_dir=None, lora_scale=1.0):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        safety_checker=None,
        local_files_only=LOCAL_FILES_ONLY,
    ).to(device)

    pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        pipe.scheduler.config,
        algorithm_type="dpmsolver++",
        use_karras_sigmas=True,
    )

    if int(pipe.unet.config.in_channels) != 9:
        raise RuntimeError("Wrong model loaded. Stable Diffusion inpainting must use a 9-channel UNet.")

    if lora_dir is not None:
        lora_dir = Path(lora_dir)
        weights_path = lora_dir / "pytorch_lora_weights.safetensors"
        if not weights_path.exists():
            raise FileNotFoundError(f"LoRA weights not found: {weights_path}")

        pipe.load_lora_weights(str(lora_dir), adapter_name="selected_lora")
        if hasattr(pipe, "set_adapters"):
            pipe.set_adapters(["selected_lora"], adapter_weights=[lora_scale])

    if device == "cuda":
        pipe.enable_attention_slicing()
        pipe.vae.enable_slicing()

    pipe.set_progress_bar_config(disable=False)
    return pipe, device


if torch.cuda.is_available():
    torch.cuda.empty_cache()

lora_pipe, lora_device = load_base_pipe_with_optional_lora(LORA_DIR, LORA_SCALE)
print("device:", lora_device)
print("pipeline:", lora_pipe.__class__.__name__)
print("unet input channels:", lora_pipe.unet.config.in_channels)
print("lora scale:", LORA_SCALE)

In [ ]:
lora_generator = torch.Generator(device=lora_device).manual_seed(RUN_SEED)

lora_result = lora_pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=test_image,
    mask_image=test_mask,
    height=RESOLUTION,
    width=RESOLUTION,
    strength=STRENGTH,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=lora_generator,
).images[0]

lora_result.save(OUTPUT_DIR / "latest_lora_result.png")
lora_result.save(OUTPUT_DIR / f"lora_result_{RUN_ID}.png")
lora_result

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
items = [
    ("input", test_image, None),
    ("mask", test_mask, "gray"),
    ("repaint area", repaint_preview, None),
    ("base", result, None),
    ("lora", lora_result, None),
]

for axis, (title, item, cmap) in zip(axes, items):
    axis.imshow(item, cmap=cmap)
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
base_array = np.asarray(result.convert("RGB"), dtype=np.int16)
lora_array = np.asarray(lora_result.convert("RGB"), dtype=np.int16)
print("base vs lora mean abs diff:", float(np.abs(base_array - lora_array).mean()))
print("saved latest LoRA result to:", OUTPUT_DIR / "latest_lora_result.png")
print("saved run LoRA result to:", OUTPUT_DIR / f"lora_result_{RUN_ID}.png")

## Potential problems, if results look wrong anywhere in the code.

Check these before changing any app code:

- If the wrong area changes, toggle `INVERT_MASK` and rerun from the inspection cells.
- If the mask coverage is tiny or huge, the mask image itself is probably wrong.
- If the source image and mask sizes differ, verify they are meant to align before resizing.
- If the image looks stretched, the source is not square and this simple test resized it to 512 x 512. 
- If the base model works here, the issue is probably in the app path rather than Stable Diffusion itself.
